### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [15]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pathlib import Path

In [17]:
### Read all PDF files in the current directory



def process_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and return a list of text chunks."""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}.")
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...  {pdf_file.name}")
        
        try:
            # Try using PyMuPDFLoader first
            loader = PyPDFLoader(pdf_file)
            documents = loader.load()
        except Exception as e:
            print(f"PyMuPDFLoader failed for {pdf_file} with error: {e}. Trying PyPDFLoader...")
            try:
                # Fallback to PyPDFLoader
                loader = PyMuPDFLoader(pdf_file)
                documents = loader.load()
            except Exception as e:
                print(f"PyPDFLoader also failed for {pdf_file} with error: {e}. Skipping this file.")
                continue
        
        all_documents.extend(documents)
        print(f"loaded {len(documents)} pages.")

    
    return all_documents


# Process all PDFs in the current directory
all_pdf_documents = process_pdfs("../data")

Found 3 PDF files in ../data.
Processing ../data/pdf/iStatementWorksheet.pdf...  iStatementWorksheet.pdf
loaded 2 pages.
Processing ../data/pdf/FocusPlanWorksheet.pdf...  FocusPlanWorksheet.pdf
loaded 3 pages.
Processing ../data/pdf/ADHDWorksheet.pdf...  ADHDWorksheet.pdf
loaded 1 pages.


In [20]:
### Text splitting get into chucks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split documents {len(documents)} into {len(split_docs)} chunks.")
    
    if split_docs:
        print(f"First chunk preview: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    
    return split_docs

In [28]:
chunks=split_documents(all_pdf_documents)

chunks

Split documents 6 into 8 chunks.
First chunk preview: "I" Statements 
Worksheet
Mentalyc IncExplore Secure AI-Powered Progress Note Automation!
"I" statements are a form of assertive communication that express your thoughts,
feelings, and needs in a clea...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-07-06T21:48:31+00:00', 'title': 'Mentalyc Cheatsheets', 'moddate': '2024-07-06T21:48:30+00:00', 'keywords': 'DAGJDqOdZO0,BAFXld6g6w4', 'author': 'Eseosa Osayimwen', 'source': '../data/pdf/iStatementWorksheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-07-06T21:48:31+00:00', 'title': 'Mentalyc Cheatsheets', 'moddate': '2024-07-06T21:48:30+00:00', 'keywords': 'DAGJDqOdZO0,BAFXld6g6w4', 'author': 'Eseosa Osayimwen', 'source': '../data/pdf/iStatementWorksheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='"I" Statements \nWorksheet\nMentalyc IncExplore Secure AI-Powered Progress Note Automation!\n"I" statements are a form of assertive communication that express your thoughts,\nfeelings, and needs in a clear and respectful manner. They focus on personal\nexperiences and avoid blaming or accusing others.\nWhy Use "I" Statements?\n"I" statements help you express emotions in a direct and non-confrontational\nmanner.\nUsing "I" statements enhances mutual understanding and promotes active\nlistening.\nThey minimize defensiveness and prevent others from feeling attacked.\n"I" statements contribute to open, respectful communication, fosterin

### Embedding and VectorStoreDB

In [29]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/newapple/Documents/GitHub/rag-pipeline/server/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        
        Initialize the embedding manager with a specified model.
        
        Args: 
            model_name (str): HuggingFace model name for sentence embeddings.
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if not self.model:
            raise ValueError("Model not loaded.")
        
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings for {len(texts)} texts.")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
        
## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loaded embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14670.88it/s]


Model loaded successfully. embedding dimension: 384


### Vector Store

In [31]:
class VectorStore: 
    """A simple vector store using ChromaDB for storing document embeddings."""
    
    def __init__(self, collection_name: str = "pdf_chunks", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store and create a ChromaDB collection.
        
        Args:
            collection_name (str): Name of the ChromaDB collection to use.
            persist_directory (str): Directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent Chroma client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document chunks and their embeddings"}
            )
            
            print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory} with collection count {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise
        
        
    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """Add documents and their embeddings to the ChromaDB collection."""
        
        if not self.collection:
            raise ValueError("ChromaDB collection not initialized.")
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        # prepare data for insertion to chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )   
              
            print(f"Added {len(documents)} documents to the vector store with")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to ChromaDB: {e}")
            raise
        
vectorStore = VectorStore()

vectorStore
    

Initialized ChromaDB collection: pdf_chunks at ../data/vector_store with collection count 0


In [33]:
chunks

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-07-06T21:48:31+00:00', 'title': 'Mentalyc Cheatsheets', 'moddate': '2024-07-06T21:48:30+00:00', 'keywords': 'DAGJDqOdZO0,BAFXld6g6w4', 'author': 'Eseosa Osayimwen', 'source': '../data/pdf/iStatementWorksheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='"I" Statements \nWorksheet\nMentalyc IncExplore Secure AI-Powered Progress Note Automation!\n"I" statements are a form of assertive communication that express your thoughts,\nfeelings, and needs in a clear and respectful manner. They focus on personal\nexperiences and avoid blaming or accusing others.\nWhy Use "I" Statements?\n"I" statements help you express emotions in a direct and non-confrontational\nmanner.\nUsing "I" statements enhances mutual understanding and promotes active\nlistening.\nThey minimize defensiveness and prevent others from feeling attacked.\n"I" statements contribute to open, respectful communication, fosterin

In [40]:
### convert text to embeddings
text = [doc.page_content for doc in chunks]

## Generate embeddings
embeddings = embedding_manager.generate_embeddings(text)

## store in the vector DB
vectorStore.add_documents(chunks, embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.07it/s]

Generated embeddings for 8 texts.
Added 8 documents to the vector store with
Total documents in collection: 16


### Retriever Pipeline From VectorStore

In [83]:
class RAGRetriever:
    """Retriever for fetching relevant document chunks based on query embeddings."""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever with a vector store and embedding manager.
        
        Args:
            vector_store (VectorStore): The vector store instance to query.
            embedding_manager (EmbeddingManager): The embedding manager to generate query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant document chunks based on the query."""
        
        print(f"end=Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold} ")
        
        try:
            # Generate embedding for the query
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]
            
            # Fetch all documents and their embeddings from the vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            print(f"Retrieved {len(results['documents'][0]) if results['documents'] else 0} documents from vector store.")
            
            #process results
            retrieved_docs = []
            
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (Chroma uses cosine distance)
                    
                    # similarity_score = 1 - distance
                    similarity_score = 1 - distance if distance <= 1 else 0
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                        
            print(f"Retrieved {len(retrieved_docs)} documents for the query.")
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise
        

rag_retriever = RAGRetriever(vector_store=vectorStore, embedding_manager=embedding_manager)

In [84]:
rag_retriever.retrieve("what is respectful communication?")

end=Retrieving documents for query: 'what is respectful communication?' with top_k=5 and score_threshold=0.0 


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.18it/s]

Generated embeddings for 1 texts.
Retrieved 5 documents from vector store.
Retrieved 5 documents for the query.


[{'id': 'doc_c1865acc_0',
  'content': '"I" Statements \nWorksheet\nMentalyc IncExplore Secure AI-Powered Progress Note Automation!\n"I" statements are a form of assertive communication that express your thoughts,\nfeelings, and needs in a clear and respectful manner. They focus on personal\nexperiences and avoid blaming or accusing others.\nWhy Use "I" Statements?\n"I" statements help you express emotions in a direct and non-confrontational\nmanner.\nUsing "I" statements enhances mutual understanding and promotes active\nlistening.\nThey minimize defensiveness and prevent others from feeling attacked.\n"I" statements contribute to open, respectful communication, fostering positive\nrelationships.\n"You never listen to me when I talk about my feelings."\nExamples of Blaming vs. "I" Statements\nBlaming Statement\n"I" Statement "I feel frustrated when I don\'t feel heard when talking\nabout my feelings."\nBlaming Statement "You always leave the dishes in the sink for me to\nclean up."',
